In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/q3_retail_promotions.csv")

df["transaction_date"] = pd.to_datetime(df["transaction_date"])

df["year"] = df["transaction_date"].dt.year
df["month"] = df["transaction_date"].dt.month
df["day_of_week"] = df["transaction_date"].dt.dayofweek
df["is_month_end"] = (df["transaction_date"].dt.day >= 25).astype(int)

df.head()

Date features were extracted from transaction_date to capture time-based sales patterns. The is_month_end feature was created because customer buying behaviour may change near the end of the month.

In [ ]:
df = df.sort_values("transaction_date")

split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

X_train = train_df.drop(["items_sold", "transaction_date"], axis=1)
y_train = train_df["items_sold"]

X_test = test_df.drop(["items_sold", "transaction_date"], axis=1)
y_test = test_df["items_sold"]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

A random split is inappropriate for time-ordered data because it can cause data leakage, where future records are used to train the model. A temporal split better reflects real-world forecasting, where past data is used to predict future sales.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = ["promotion_type", "location_type", "store_size"]

numerical_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numerical_features)
    ]
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42))
])

lr_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

lr_pred = lr_pipeline.predict(X_test)
rf_pred = rf_pipeline.predict(X_test)

In [ ]:
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_mae = mean_absolute_error(y_test, lr_pred)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae = mean_absolute_error(y_test, rf_pred)

results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest Regressor"],
    "RMSE": [lr_rmse, rf_rmse],
    "MAE": [lr_mae, rf_mae]
})

results

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, lr_pred)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()])
plt.xlabel("Actual items_sold")
plt.ylabel("Predicted items_sold")
plt.title("Linear Regression: Predicted vs Actual")
plt.show()

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, rf_pred)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()])
plt.xlabel("Actual items_sold")
plt.ylabel("Predicted items_sold")
plt.title("Random Forest: Predicted vs Actual")
plt.show()

In [ ]:
feature_names = rf_pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = rf_pipeline.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

importance_df.head(10)

In [ ]:
top_5_features = importance_df.head(5)
top_5_features

The top five most influential features were identified using feature importance from the Random Forest model. These features had the strongest impact on predicting items_sold.